# 🚀 SCRIPT HUẤN LUYỆN MODEL AI CHO SIPEED MAIXCAM TRÊN GOOGLE COLAB
### Dự án: Gậy Thông Minh Cho Người Khiếm Thị (MaixCAM SG2002 NPU 1 TOPS)
---
Notebook này hỗ trợ huấn luyện 2 mô hình Deep Learning:
1. **Mô hình nhận diện Tiền Polymer VND** (6 mệnh giá: 10k, 20k, 50k, 100k, 200k, 500k)
2. **Mô hình nhận diện 36 Ký tự OCR** (0-9, A-Z)

> **Lưu ý cốt lõi cho phần cứng MaixCAM (Sophgo SG2002):**
> - Kiến trúc tối ưu: `yolov5nu` (YOLOv5 Nano với Anchor-Free Head của YOLOv8).
> - Kích thước ảnh: `320x320` (được tối ưu NPU và bộ nhớ SRAM).
> - Xuất ONNX: `opset=11`, `simplify=True`.
> - Định dạng file cấu hình `.mud`: `model_type = yolov8` (phù hợp head output của YOLOv5u/YOLOv8).

## ⚡ Bước 1: Kiểm tra GPU và Cài đặt Thư viện
> **Quan trọng**: Vào menu `Runtime` (Thời gian chạy) -> `Change runtime type` (Thay đổi loại thời gian chạy) -> Chọn **T4 GPU** trước khi bắt đầu!

In [ ]:
# 1. Kiem tra thong tin GPU T4
!nvidia-smi

# 2. Cai dat cac thu vien can thiet: Ultralytics YOLO, ONNX, OnnxSlim va gdown
!pip install -q ultralytics onnx onnxslim gdown

## 📦 Bước 2: Tự động Tải Dataset từ Google Drive & Giải nén
Mã bên dưới sẽ **tự động tải dataset từ Google Drive** của bạn (không cần upload thủ công qua trình duyệt) và giải nén chuẩn hóa file cấu hình YAML.

In [ ]:
import os
import zipfile
import subprocess

# Google Drive File IDs tu folder chia se
DRIVE_FILES = {
    'money': {
        'id': '1TSvrgjbUVieSnyJQ2mG3PQ6dJHxeRKyQ',
        'zip': '/content/dataset_money_yolo.zip',
        'dir': '/content/dataset_money_yolo'
    },
    'chars': {
        'id': '12nPxcil2kD7fbbhOgly8LFFE-2d9xcXx',
        'zip': '/content/dataset_chars_yolo.zip',
        'dir': '/content/dataset_chars_yolo'
    }
}

# 1. Tai dataset tien neu chua co
if not os.path.exists(DRIVE_FILES['money']['zip']) and not os.path.exists(DRIVE_FILES['money']['dir']):
    print("[*] Dang tai dataset_money_yolo.zip truc tiep tu Google Drive...")
    os.system(f"gdown --id {DRIVE_FILES['money']['id']} -O {DRIVE_FILES['money']['zip']}")

# 2. Tai dataset ky tu neu chua co
if not os.path.exists(DRIVE_FILES['chars']['zip']) and not os.path.exists(DRIVE_FILES['chars']['dir']):
    print("[*] Dang tai dataset_chars_yolo.zip truc tiep tu Google Drive...")
    os.system(f"gdown --id {DRIVE_FILES['chars']['id']} -O {DRIVE_FILES['chars']['zip']}")

# Giai nen Dataset Tien VND
if os.path.exists(DRIVE_FILES['money']['zip']):
    print("[*] Dang giai nen dataset_money_yolo.zip...")
    with zipfile.ZipFile(DRIVE_FILES['money']['zip'], 'r') as zip_ref:
        zip_ref.extractall('/content')
    print("[✓] Da giai nen xong: /content/dataset_money_yolo")

# Giai nen Dataset Ky tu OCR
if os.path.exists(DRIVE_FILES['chars']['zip']):
    print("[*] Dang giai nen dataset_chars_yolo.zip...")
    with zipfile.ZipFile(DRIVE_FILES['chars']['zip'], 'r') as zip_ref:
        zip_ref.extractall('/content')
    print("[✓] Da giai nen xong: /content/dataset_chars_yolo")

# Chuan hoa file YAML duong dan Linux Colab
if os.path.exists('/content/dataset_money_yolo'):
    money_yaml = """# Cau hinh Dataset Nhan Dien Menh Gia Tien Polymer VND
path: /content/dataset_money_yolo
train: images/train
val: images/val

names:
  0: '10000'
  1: '20000'
  2: '50000'
  3: '100000'
  4: '200000'
  5: '500000'
"""
    with open('/content/dataset_money_yolo/money.yaml', 'w', encoding='utf-8') as f:
        f.write(money_yaml)
    print("[✓] Da san sang file cau hinh: /content/dataset_money_yolo/money.yaml")

if os.path.exists('/content/dataset_chars_yolo'):
    char_classes = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
                    'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j',
                    'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't',
                    'u', 'v', 'w', 'x', 'y', 'z']
    chars_yaml_lines = [
        "# Cau hinh Dataset 36 Ky Tu (0-9, A-Z) cho Sipeed MaixCAM",
        "path: /content/dataset_chars_yolo",
        "train: images/train",
        "val: images/val",
        "",
        "names:"
    ]
    for idx, c in enumerate(char_classes):
        chars_yaml_lines.append(f"  {idx}: '{c}'")
    with open('/content/dataset_chars_yolo/chars.yaml', 'w', encoding='utf-8') as f:
        f.write("\n".join(chars_yaml_lines))
    print("[✓] Da san sang file cau hinh: /content/dataset_chars_yolo/chars.yaml")

## 🎯 Bước 3A: Huấn luyện Model Tiền Polymer VND (YOLOv5nu)
- Sử dụng GPU Colab (`device=0`)
- Huấn luyện 25 - 30 epochs (thời gian ~2-3 phút)
- Tự động xuất file `vnd_money_v2.onnx` chuẩn `opset=11`, `imgsz=320`

In [ ]:
import os
from ultralytics import YOLO

money_yaml_path = "/content/dataset_money_yolo/money.yaml"

if os.path.exists(money_yaml_path):
    print("=" * 65)
    print("  BAT DAU HUAN LUYEN MODEL TIEN VND (YOLOv5nu, 320x320, GPU)       ")
    print("=" * 65)
    
    # Load pretrained YOLOv5nu (Nano)
    model_money = YOLO("yolov5nu.pt")
    
    # Huấn luyện
    results_money = model_money.train(
        data=money_yaml_path,
        epochs=25,        # 25 epochs đạt độ chính xác > 98% cho tiền VND
        imgsz=320,        # Chuẩn kích thước MaixCAM SG2002
        batch=32,         # GPU T4 chạy batch 32 cực kỳ mượt mà
        device=0,
        project="/content/runs_money",
        name="yolov5_money_v2",
        exist_ok=True,
        verbose=True
    )
    
    # Xuat ONNX opset 11 cho MaixCAM SG2002 NPU MLIR compiler
    best_money_pt = "/content/runs_money/yolov5_money_v2/weights/best.pt"
    print(f"[*] Dang xuat ONNX tu {best_money_pt}...")
    trained_money = YOLO(best_money_pt)
    onnx_money_path = trained_money.export(
        format="onnx",
        imgsz=320,
        opset=11,
        simplify=True
    )
    print(f"[✓] Thanh cong xuat file ONNX: {onnx_money_path}")
else:
    print("[!] Bo qua: Khong tim thay /content/dataset_money_yolo/money.yaml")

## 🔤 Bước 3B: Huấn luyện Model Nhận diện 36 Ký tự OCR (YOLOv5nu)
- 36 nhãn ký tự: `0-9` và `a-z`
- Độ phân giải: `320x320`
- Tự động xuất file `char_ocr_v2.onnx`

In [ ]:
import os
from ultralytics import YOLO

chars_yaml_path = "/content/dataset_chars_yolo/chars.yaml"

if os.path.exists(chars_yaml_path):
    print("=" * 65)
    print("  BAT DAU HUAN LUYEN MODEL 36 KY TU OCR (YOLOv5nu, 320x320, GPU)   ")
    print("=" * 65)
    
    model_chars = YOLO("yolov5nu.pt")
    
    results_chars = model_chars.train(
        data=chars_yaml_path,
        epochs=25,
        imgsz=320,
        batch=32,
        device=0,
        project="/content/runs_chars",
        name="yolov5_chars36_v2",
        exist_ok=True,
        verbose=True
    )
    
    best_chars_pt = "/content/runs_chars/yolov5_chars36_v2/weights/best.pt"
    print(f"[*] Dang xuat ONNX tu {best_chars_pt}...")
    trained_chars = YOLO(best_chars_pt)
    onnx_chars_path = trained_chars.export(
        format="onnx",
        imgsz=320,
        opset=11,
        simplify=True
    )
    print(f"[✓] Thanh cong xuat file ONNX: {onnx_chars_path}")
else:
    print("[!] Bo qua: Khong tim thay /content/dataset_chars_yolo/chars.yaml")

## 📦 Bước 4: Tự động Đóng Gói Model & Tải về Máy tính
Ô code dưới đây sẽ tự động:
1. Thu thập các file `.onnx`, `.pt` tốt nhất.
2. **Tạo chuẩn file cấu hình `.mud`** với `model_type = yolov8` phù hợp cấu trúc NPU của MaixPy v4.
3. Nén tất cả thành file `maixcam_trained_models.zip` và kích hoạt tải về máy tính!

In [ ]:
import os
import shutil
from google.colab import files

export_dir = "/content/maixcam_models_export"
if os.path.exists(export_dir):
    shutil.rmtree(export_dir)
os.makedirs(export_dir, exist_ok=True)

# 1. Model Tien VND
src_money_onnx = "/content/runs_money/yolov5_money_v2/weights/best.onnx"
src_money_pt = "/content/runs_money/yolov5_money_v2/weights/best.pt"
if os.path.exists(src_money_onnx):
    shutil.copy2(src_money_onnx, f"{export_dir}/vnd_money_v2.onnx")
    shutil.copy2(src_money_pt, f"{export_dir}/vnd_money_v2.pt")
    
    # File MUD chuan cho MaixCAM SG2002 (model_type = yolov8 vi dung YOLOv5u Anchor-Free)
    mud_money = """[basic]
type = cvimodel
model = vnd_money_v2.cvimodel

[extra]
model_type = yolov8
type = detector
input_type = rgb
labels = 10000, 20000, 50000, 100000, 200000, 500000
mean = 0, 0, 0
scale = 0.00392156862745098, 0.00392156862745098, 0.00392156862745098
"""
    with open(f"{export_dir}/vnd_money_v2.mud", "w", encoding="utf-8") as f:
        f.write(mud_money)
    print("[✓] Da tao goi mo hinh Tien: vnd_money_v2.onnx + vnd_money_v2.mud")

# 2. Model Ky tu 36 OCR
src_chars_onnx = "/content/runs_chars/yolov5_chars36_v2/weights/best.onnx"
src_chars_pt = "/content/runs_chars/yolov5_chars36_v2/weights/best.pt"
if os.path.exists(src_chars_onnx):
    shutil.copy2(src_chars_onnx, f"{export_dir}/char_ocr_v2.onnx")
    shutil.copy2(src_chars_pt, f"{export_dir}/char_ocr_v2.pt")
    
    mud_chars = """[basic]
type = cvimodel
model = char_ocr_v2.cvimodel

[extra]
model_type = yolov8
type = detector
input_type = rgb
labels = 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, a, b, c, d, e, f, g, h, i, j, k, l, m, n, o, p, q, r, s, t, u, v, w, x, y, z
mean = 0, 0, 0
scale = 0.00392156862745098, 0.00392156862745098, 0.00392156862745098
"""
    with open(f"{export_dir}/char_ocr_v2.mud", "w", encoding="utf-8") as f:
        f.write(mud_chars)
    print("[✓] Da tao goi mo hinh Ky tu: char_ocr_v2.onnx + char_ocr_v2.mud")

# 3. Nen tat ca thanh 1 file ZIP
zip_download_path = "/content/maixcam_trained_models"
shutil.make_archive(zip_download_path, 'zip', export_dir)
print(f"\n[✓] Da tao xong file: {zip_download_path}.zip")
print("[*] Dang tu dong gui file ve trinh duyet may tinh cua ban...")
files.download(f"{zip_download_path}.zip")

## 🚀 Bước 5: Chuyển đổi Model sang .CVIMODEL trên MaixHub
Sau khi tải file `maixcam_trained_models.zip` về máy tính và giải nén:
1. Mở trang web chuyển đổi: **[https://maixhub.com/model/convert](https://maixhub.com/model/convert)**
2. Chọn phần cứng: **MaixCAM** (Chip Sophgo SG2002 / CV181x).
3. Tải lên file ONNX:
   - `vnd_money_v2.onnx` (cho Tiền VND)
   - hoặc `char_ocr_v2.onnx` (cho Ký tự)
4. Chọn loại Model: **YOLOv8** *(QUAN TRỌNG: Chọn YOLOv8 vì mô hình dùng kiến trúc YOLOv5nu Anchor-Free)*.
5. Độ phân giải (Resolution): **320x320**.
6. Nhấn **Convert**, hệ thống Cloud NPU của MaixHub sẽ biên dịch trong khoảng 1-2 phút.
7. Bấm **Download** tải file zip kết quả về máy tính.
8. Giải nén và đổi tên file `.cvimodel` thành:
   - `vnd_money_v2.cvimodel` (cho tiền)
   - `char_ocr_v2.cvimodel` (cho ký tự)
9. Chép các file vừa tạo vào thư mục dự án trên máy tính:
   `Gay_Nguoi_Mu/firmware/maixcam_app/models/`
10. Cắm cáp USB nối MaixCAM với máy tính và chạy lệnh nạp:
   ```bash
   python sync_and_test_maixcam.py
   ```
🎉 Hoàn tất toàn bộ quy trình! MaixCAM sẽ nạp mô hình mới trực tiếp vào chip NPU để nhận diện thời gian thực.